# TML — Unified A100 Compute
One Colab session, VPS-controlled. RapidOCR and Layout run in isolated environments and are switched by the VPS controller. Default YEAR=1904.


In [ ]:
YEAR=1904
RAPID_WORKERS=12
RAPID_DOWNLOADERS=8
LAYOUT_WORKERS=4
LAYOUT_DOWNLOADERS=8
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
print('CONFIG',YEAR,'Rapid',RAPID_WORKERS,'Layout',LAYOUT_WORKERS,'BASE',BASE,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
RAPID_ENV='/content/tml-rapid-env'
LAYOUT_ENV='/content/tml-layout-env'
RAPID_PY=f'{RAPID_ENV}/bin/python'
LAYOUT_PY=f'{LAYOUT_ENV}/bin/python'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv','paramiko>=3.5,<4'],check=True)
UV=shutil.which('uv'); assert UV
print('SETUP RapidOCR isolated env',flush=True)
if not os.path.exists(RAPID_PY): subprocess.run([UV,'venv','--seed',RAPID_ENV],check=True)
subprocess.run([RAPID_PY,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements.txt'],check=True)
subprocess.run([RAPID_PY,'-c',"import onnxruntime as o; p=o.get_available_providers(); print('RAPID_ENV_READY',o.__version__,p); assert 'CUDAExecutionProvider' in p"],check=True)
print('SETUP Layout isolated Python 3.12 env',flush=True)
subprocess.run([UV,'python','install','3.12'],check=True)
if not os.path.exists(LAYOUT_PY): subprocess.run([UV,'venv','--seed','--python','3.12',LAYOUT_ENV],check=True)
PADDLE_URL='https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'
PADDLE_WHL='/content/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'
EXPECTED=1890365820
need=subprocess.run([LAYOUT_PY,'-c',"import paddle,sys; sys.exit(0 if paddle.__version__=='3.2.0' and paddle.device.is_compiled_with_cuda() else 1)"]).returncode!=0
if need:
 if not os.path.exists(PADDLE_WHL) or os.path.getsize(PADDLE_WHL)!=EXPECTED: subprocess.run(['curl','-L','--fail','--retry','5','-C','-','--progress-bar','-o',PADDLE_WHL,PADDLE_URL],check=True)
 subprocess.run([LAYOUT_PY,'-m','pip','install','--progress-bar','on',PADDLE_WHL],check=True)
subprocess.run([LAYOUT_PY,'-m','pip','install','--progress-bar','on','paddlex==3.7.2','paddleocr==3.7.0','opencv-contrib-python==4.10.0.84','paramiko>=3.5,<4'],check=True)
subprocess.run([LAYOUT_PY,'-c',"import paddle,paddlex,cv2; print('LAYOUT_ENV_READY',paddle.__version__,paddlex.__version__,cv2.__version__,'CUDA',paddle.device.is_compiled_with_cuda())"],check=True)
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'
open(KEY_FILE,'wb').write(data)
os.chmod(KEY_FILE,0o600)
print('KEY_READY',name,flush=True)


In [ ]:
import importlib.util,subprocess,sys
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip()
print('CODE',commit,flush=True)
path=f'{REPO}/colab/compute_supervisor.py'
spec=importlib.util.spec_from_file_location('tml_compute_supervisor',path)
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
argv=['compute_supervisor.py','--year',str(YEAR),'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--repo',REPO,'--rapid-python',RAPID_PY,'--layout-python',LAYOUT_PY,'--rapid-workers',str(RAPID_WORKERS),'--rapid-downloaders',str(RAPID_DOWNLOADERS),'--layout-workers',str(LAYOUT_WORKERS),'--layout-downloaders',str(LAYOUT_DOWNLOADERS),'--poll','10']
print('STARTING_UNIFIED_COMPUTE_SUPERVISOR_IN_PROCESS',flush=True)
old_argv=sys.argv[:]; sys.argv=argv
try: mod.main()
finally: sys.argv=old_argv
